# Setting up the model

 Based on Custom Prediction Routines service: https://cloud.google.com/vertex-ai/docs/predictions/custom-prediction-routines

A model has a few parts
* Predictor class - handlers the model loading and running
* Handler class - handles the fast API request
* Docker env - from a large selection


GCP Prebuiuld containers
* Docker Predict env: https://console.cloud.google.com/artifacts/docker/vertex-ai/us/prediction
    * https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers
* Maybe Dockerfile code?: https://github.com/huggingface/Google-Cloud-Containers/blob/main/containers/tgi/gpu/2.4.0/Dockerfile
* Base Images: https://console.cloud.google.com/artifacts/docker/deeplearning-platform-release/us/gcr.io
* GCP hugging face: https://cloud.google.com/deep-learning-containers/docs/choosing-container#hugging-face
*

# Setting up GCP
1. Create the registry in GCP
2. authenticate gcloud
3. authenticate docker with gcloud
4. Grant the vertex service account access to artifact registry
    1. click `Include Google-provided role grants`
    1. find `gcp-sa-aiplatform` and grant Artifact Registry writer to it

# Working Sample
https://huggingface.co/blog/alvarobartt/deploy-from-hub-to-vertex-ai


## Basic Model
1. Deploys the model in `custom_model` dir
    1. predictor.py
    1. requirements.txt, generated from the `custom-model` group in `notebooks/pyproject.toml`
2. Deploys to
    1. Artifact registry
    1. Model Registry
    1. Vertex Endpoints
1. Deploys on a machine that costs $1.5 an hour

In [ ]:
# @title Loads the shared env vars from the root directory
from pathlib import Path

# Allows running async functions
import nest_asyncio
from nb_settings import settings

nest_asyncio.apply()

model = settings.llm_model_name
print(f"Using the model: {model}")
current_dir = Path.cwd()

In [ ]:
# @title Optional Auth
!gcloud auth configure-docker us-central1-docker.pkg.dev

In [ ]:
from google.cloud import aiplatform
from google.cloud.aiplatform.prediction import LocalModel

IMAGE_NAME = "predict-image"
MODEL_DISPLAY_NAME = "My Model"

aiplatform.init(
    project=settings.project_id,
    location=settings.region,
)

src_dir = current_dir / "custom_model"
req_path = src_dir / "requirements.txt"

# Regenerate the container's requirements from notebooks/pyproject.toml
!make -C {current_dir} requirements

# From huggingface, what is this? https://huggingface.co/blog/alvarobartt/deploy-from-hub-to-vertex-ai
base_image = "alvarobartt/torch-gpu:py310-cu12.3-torch-2.2.0"
# A dep error with numpy being over >=2.0
# base_image = "gcr.io/deeplearning-platform-release/pytorch-gpu.2-4.py310:latest"
# base_image = "gcr.io/deeplearning-platform-release/huggingface-pytorch-inference-cu121.2-3.transformers.4-48.ubuntu2204.py311:latest"

REPOSITORY = "my-model-repo"

image_dest = (
    f"{settings.region}-docker.pkg.dev/{settings.project_id}/{REPOSITORY}/{IMAGE_NAME}"
)

%cd $src_dir

In [ ]:
# @title run if you change the model

import importlib

import predictor

importlib.reload(predictor)

In [ ]:
# @title Builds Docker Image
# Creates a docker image named for Google artifact reg

import logging

from predictor import ImagePredictor

logging.basicConfig(level=logging.INFO)

# I dont know how to set envs!!!
# Add to file venv/lib/python3.12/site-packages/google/cloud/aiplatform/prediction/local_model.py
# Add on Line 372
#   "VERTEX_CPR_WEB_CONCURRENCY": "1",

local_model = LocalModel.build_cpr_model(
    str(src_dir),
    image_dest,
    predictor=ImagePredictor,
    # handler={HANDLER_CLASS},
    requirements_path=str(req_path),
    base_image=base_image,
    platform="linux/amd64",  # Vertex runs on linux
)

In [ ]:
# @title Uploads Image to GCP
# Create the repo at: https://console.cloud.google.com/artifacts/browse/<MY_PROJECT>
# Log into docker: gcloud auth configure-docker us-central1-docker.pkg.dev

local_model.push_image()

In [ ]:
# @title Uploads the Model.
# You can upload by local_model var or by image URI
# ISSUE WARNING
# Do IMAGE URI because you can set the env to prevent the multi worker issue

vertex_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    description=" Base Model: alvarobartt",
    serving_container_image_uri=image_dest,
    serving_container_environment_variables={
        "VERTEX_CPR_WEB_CONCURRENCY": "1"  # Default 1 worker per core.
    },
    project=settings.project_id,
)
print(vertex_model.resource_name)

In [ ]:
# @title Loads existing endpoint

endpoint_id = "3935078448531243008"
endpoint_name = f"projects/{settings.project_id}/locations/{settings.region}/endpoints/{endpoint_id}"
endpoint = aiplatform.Endpoint(endpoint_name=endpoint_name)

In [ ]:
# @title Deploys Model
# Creates an endpoint and deploys the model to an expensive machine
# You undeploy the model and leave the endpoint to not get charged
# Price: "a2-highgpu-1g" - $4 an hour
# https://cloud.google.com/compute/all-pricing?hl=en

# machine_type="n1-standard-8"
# accelerator_type="NVIDIA_TESLA_V100"
# machine_type="n1-standard-4"
# accelerator_type="NVIDIA_TESLA_T4"

# Need for SD models
machine_type = "a2-highgpu-1g"
accelerator_type = "NVIDIA_TESLA_A100"

accelerator_count = 1

endpoint = vertex_model.deploy(
    endpoint=endpoint,
    machine_type=machine_type,
    service_account=f"{settings.project_id}@appspot.gserviceaccount.com",
    accelerator_type=accelerator_type,
    accelerator_count=accelerator_count,
)
print(endpoint.name)

# Manage and Test Deployed Models
These commands load the models and endpoint configs


In [ ]:
# Run this if you dont have the model object. ID can be grabbed from model url
# SKIP if you already have `endpoints` loaded

MODEL_ID = "4545113887900434432"
model_name = (
    f"projects/{settings.project_id}/locations/{settings.region}/models/{MODEL_ID}"
)
# model_name = ""
vertex_model = aiplatform.Model(model_name=model_name)

In [ ]:
# Run this to locate the endpoint object
# SKIP if you already have `endpoints` loaded

endpoint_id = "3935078448531243008"
endpoint_name = f"projects/{settings.project_id}/locations/{settings.region}/endpoints/{endpoint_id}"
endpoint = aiplatform.Endpoint(endpoint_name=endpoint_name)

In [ ]:
# @title Deploy existing endpoint

model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="your-deployment-name",
    machine_type="n1-standard-8",
)

In [ ]:
# @title Call the endpoint.
# https://cloud.google.com/vertex-ai/docs/reference/rest/v1/projects.locations.endpoints/predict#request-body
data = {
    "prompt": "A glowing cube floating over a futuristic city.",
    "lora": "style-b",
}

instances = [data]
resp = endpoint.predict(instances=[data])

In [ ]:
print(resp.metadata)

In [ ]:
# @title Saves Image
# Saves the endpoint image response

import base64

img_str = resp.predictions[0]["image"]
with open("output.png", "wb") as f:
    f.write(base64.b64decode(img_str))

In [ ]:
# @title Undeploy a Model
# Important to save $$$$

endpoint.undeploy(deployed_model_id="2360945034439688192")

In [ ]:
# @title delete endpoint
# Not important

endpoint.delete()

In [ ]:
# @title Create Endpoint
# Not needed if you deploy a model directly

endpoint = aiplatform.Endpoint.create(
    display_name="My Model",
    project=settings.project_id,
    location=settings.region,
)

# Local Dev
Loads the model locally (if you have the power)

In [ ]:
# @title Local testing only
# This is for mac systems
import json

data = {"prompt": "Test", "lora": "style-a", "test": "1"}
# For mac testing

with local_model.deploy_to_local_endpoint(
    artifact_uri="./custom_model",
) as local_endpoint:
    request = {"instances": [data]}
    headers = {"Content-Type": "application/json"}
    health_check_response = local_endpoint.run_health_check()
    predict_response = local_endpoint.predict(
        request=json.dumps(request), headers=headers
    )

    local_endpoint.print_container_logs(show_all=True)

print(predict_response)
print(predict_response.json())

In [ ]:
# Deploy from local model. Needs monkey patch for workers.
# I had to run twice?
vertex_model = aiplatform.Model.upload(
    local_model=local_model,
    display_name="my-model",
    project=settings.project_id,
)
print(vertex_model.resource_name)
print(vertex_model.name)